# IVS Weekmonitor Analysis

This notebook demonstrates how to process the IVS weekmonitor data from Rijkswaterstaat, aggregate cargo by origin-destination (OD) pairs, and map these flows to the inland waterway network.

In [ ]:
import pathlib
import zipfile
import requests
import pickle
import itertools
import logging

import pandas as pd
import geopandas as gpd
import shapely.wkt
import shapely.geometry
import networkx as nx
import matplotlib.pyplot as plt

# Set up logging instead of printing
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

## 1. Download and Extract IVS Data
We download the latest weekmonitor from the Rijkswaterstaat open data portal.

In [ ]:
url = "https://downloads.rijkswaterstaatdata.nl/scheepvaart/goederenvervoer/archief/IVS_weekmonitor_31MAR2026_20260401_051939.zip"
zip_path = pathlib.Path("IVS_weekmonitor.zip")

if not zip_path.exists():
    logger.info(f"Downloading {url}...")
    r = requests.get(url)
    zip_path.write_bytes(r.content)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    csv_filename = zip_ref.namelist()[0]
    logger.info(f"Extracting {csv_filename}...")
    zip_ref.extractall(".")

logger.info("Data ready.")

## 2. Load and Aggregate OD Data
We group the cargo data by origin (`UNLO_herkomst`) and destination (`UNLO_bestemming`).

In [ ]:
ivs_df = pd.read_csv(csv_filename, sep=";")
logger.info(f"Loaded {len(ivs_df)} records.")

# Aggregate transported weight (v38_Vervoerd_gewicht) by OD pair
od_pairs = (
    ivs_df.groupby(["UNLO_herkomst", "UNLO_bestemming"])["v38_Vervoerd_gewicht"]
    .sum()
    .reset_index()
)
od_pairs = od_pairs.sort_values(by="v38_Vervoerd_gewicht", ascending=False)

logger.info("Top 5 OD Pairs by Weight:")
for idx, row in od_pairs.head(5).iterrows():
    logger.info(
        f"  {row['UNLO_herkomst']} -> {row['UNLO_bestemming']}: {row['v38_Vervoerd_gewicht']:,} kg"
    )

## 3. Load Network and Map UN/LOCODEs
We load the merged graph and map the 5-character UN/LOCODEs to network nodes.

In [ ]:
graph_path = pathlib.Path("../output/merged-graph/graph.pickle")
with graph_path.open("rb") as f:
    graph = pickle.load(f)

nodes_gdf = gpd.GeoDataFrame(graph.nodes.values(), index=graph.nodes.keys())
logger.info(f"Loaded network with {len(nodes_gdf)} nodes.")


def find_nodes_for_locode(locode):
    """Find network nodes matching the given 5-char UN/LOCODE."""
    # Match directly or by part of the ISRS locode
    mask = nodes_gdf["locode"].str.startswith(locode, na=False) | nodes_gdf[
        "locode"
    ].str.contains(locode[2:], na=False)
    return nodes_gdf[mask]


# Example: Antwerp (BEANR) to Stuttgart (DESTR)
# Note: IVS uses DESGW for Stuttgart-Gaisburg, network uses DESTR
origin_locode = "BEANR"
dest_locode = "DESTR"

origin_nodes = find_nodes_for_locode(origin_locode)
dest_nodes = find_nodes_for_locode(dest_locode)

logger.info(f"Found {len(origin_nodes)} nodes for {origin_locode}")
logger.info(f"Found {len(dest_nodes)} nodes for {dest_locode}")

## 4. Route and Visualize
We select the nearest nodes to the theoretical port centers and compute the shortest path.

In [ ]:
if not origin_nodes.empty and not dest_nodes.empty:
    # For simplicity, pick the first node from each
    start_node_id = origin_nodes.index[0]
    end_node_id = dest_nodes.index[0]

    logger.info(f"Computing path from {start_node_id} to {end_node_id}...")
    try:
        route = nx.shortest_path(graph, start_node_id, end_node_id, weight="length_m")

        # Extract geometries for visualization
        geoms = []
        for u, v in itertools.pairwise(route):
            edge_data = graph.edges[u, v]
            geom = edge_data.get("geometry")
            if isinstance(geom, str):
                geom = shapely.wkt.loads(geom)
            if geom:
                geoms.append(geom)

        route_gdf = gpd.GeoDataFrame(geometry=geoms)

        # Simple plot
        fig, ax = plt.subplots(figsize=(10, 8))
        nodes_gdf.plot(ax=ax, color="lightgrey", markersize=1, alpha=0.5)
        if not route_gdf.empty:
            route_gdf.plot(
                ax=ax,
                color="blue",
                linewidth=2,
                label=f"{origin_locode} to {dest_locode}",
            )
        ax.set_title(f"Cargo Route: {origin_locode} to {dest_locode}")
        plt.legend()
        plt.show()

    except nx.NetworkXNoPath:
        logger.error(f"No path found between {start_node_id} and {end_node_id}")
else:
    logger.warning("Could not find origin or destination nodes in the network.")